# ПР10 системы ИИ

**Цель:** реализовать метод анализа иерархий Саати (AHP) и выбрать лучшую квартиру для аренды в Москве.

В работе сравниваются 4 условные однокомнатные квартиры по 6 критериям: цена, время до центра, время до метро, площадь, состояние и время до учёбы.

Все предложения и экспертные оценки смоделированы для учебной задачи.

In [1]:
import numpy as np
import pandas as pd

## Задание 1. Реализация класса AHP

Метод Саати основан на парных сравнениях. Оценка 1 означает одинаковую важность, 3 — заметное преимущество, 5 — сильное, 9 — очень большое. Для обратного сравнения используем 1/3, 1/5 и т. д.

Класс принимает сравнения критериев и квартир, рассчитывает их веса и проверяет согласованность:

$$CI = \frac{\lambda_{max}-n}{n-1}$$


Здесь $n$ — размер матрицы, $\lambda_{max}$ — наибольшее собственное значение. Для матрицы $6 \times 6$ используем $RI = 1.24$, для $4 \times 4$ — $RI = 0.90$. Если CR < 0.1, оценки считаются достаточно согласованными.

In [2]:
class AHP:
    def __init__(self, criteria, alternatives):
        self.criteria = criteria
        self.alternatives = alternatives
        self.criteria_matrix = None
        self.alternative_matrices = {}

    def make_matrix(self, comparisons, n):
        matrix = np.ones((n, n))
        for i, j, value in comparisons:
            matrix[i, j] = value
            matrix[j, i] = 1 / value
        return matrix

    def add_criteria_comparison(self, comparisons):
        self.criteria_matrix = self.make_matrix(comparisons, len(self.criteria))

    def add_alternative_comparison(self, criterion, comparisons):
        self.alternative_matrices[criterion] = self.make_matrix(
            comparisons, len(self.alternatives)
        )

    def calculate_weights(self, matrix):
        eigenvalues, eigenvectors = np.linalg.eig(matrix)
        index = np.argmax(eigenvalues.real)
        weights = eigenvectors[:, index].real
        return weights / weights.sum()

    def check_consistency(self, matrix):
        n = len(matrix)
        eigenvalues = np.linalg.eigvals(matrix)
        lambda_max = max(eigenvalues.real)
        ci = (lambda_max - n) / (n - 1)
        ri = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12,
              6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45}
        cr = ci / ri[n] if ri[n] != 0 else 0
        return ci, cr

    def solve(self):
        criteria_weights = self.calculate_weights(self.criteria_matrix)
        local_weights = np.column_stack([
            self.calculate_weights(self.alternative_matrices[criterion])
            for criterion in self.criteria
        ])
        global_weights = local_weights @ criteria_weights
        return criteria_weights, local_weights, global_weights

## Задание 2. Построение иерархии

1. **Цель:** выбрать квартиру для аренды в Москве.
2. **Критерии:** цена, время до центра, время до метро, площадь, состояние, время до учёбы.
3. **Альтернативы:** квартира А, квартира Б, квартира В, квартира Г.

Смоделируем четыре однокомнатные квартиры.

Меньшая цена и меньшее время в пути считаются преимуществом; большая площадь и лучшее состояние — тоже.

In [3]:
criteria = ['Цена', 'До центра', 'До метро', 'Площадь', 'Состояние', 'До учёбы']
alternatives = ['Квартира А', 'Квартира Б', 'Квартира В', 'Квартира Г']

# Четыре условных предложения аренды в Москве
apartments = pd.DataFrame({
    'Аренда, руб./мес.': [60000, 88000, 70000, 75000],
    'До центра, мин': [70, 25, 40, 45],
    'До метро пешком, мин': [18, 6, 12, 8],
    'Площадь, м²': [35, 34, 39, 45],
    'Состояние': ['Обычное', 'Хорошее', 'Хорошее', 'После ремонта'],
    'До учёбы, мин': [65, 40, 25, 45]
}, index=alternatives)

print('Цель: выбрать квартиру для аренды в Москве')
print('Критерии:', ', '.join(criteria))
print('Альтернативы:', ', '.join(alternatives))
display(apartments)

Цель: выбрать квартиру для аренды в Москве
Критерии: Цена, До центра, До метро, Площадь, Состояние, До учёбы
Альтернативы: Квартира А, Квартира Б, Квартира В, Квартира Г


,"Аренда, руб./мес.","До центра, мин","До метро пешком, мин","Площадь, м²",Состояние,"До учёбы, мин"
Квартира А,60000,70,18,35,Обычное,65
Квартира Б,88000,25,6,34,Хорошее,40
Квартира В,70000,40,12,39,Хорошее,25
Квартира Г,75000,45,8,45,После ремонта,45


## Задание 3. Матрицы парных сравнений

Сначала эксперт сравнивает критерии по важности, затем — квартиры по каждому критерию. Например, (0, 1, 9) означает, что для первого эксперта цена намного важнее времени до центра. Обратное значение 1/9 программа добавит сама.

**Эксперт 1** в первую очередь хочет сэкономить на аренде. Сравнения квартир составлены с учётом таблицы, но это условные экспертные оценки, а не автоматический расчёт по ценам и минутам.

In [4]:
# Сравнения 6 критериев: цена, центр, метро, площадь, состояние, учёба
criteria_comparisons_1 = [
    (0, 1, 9), (0, 2, 4), (0, 3, 9), (0, 4, 9), (0, 5, 4),
    (1, 2, 1/2), (1, 3, 1), (1, 4, 1), (1, 5, 1/2),
    (2, 3, 2), (2, 4, 2), (2, 5, 1),
    (3, 4, 1), (3, 5, 1/2),
    (4, 5, 1/2)
]

# Сравнения квартир А, Б, В, Г по каждому критерию
apartment_comparisons = {
    'Цена': [
        (0, 1, 4), (0, 2, 2), (0, 3, 3),
        (1, 2, 1/3), (1, 3, 1/2), (2, 3, 2)
    ],
    'До центра': [
        (0, 1, 1/5), (0, 2, 1/3), (0, 3, 1/2),
        (1, 2, 3), (1, 3, 4), (2, 3, 2)
    ],
    'До метро': [
        (0, 1, 1/5), (0, 2, 1/2), (0, 3, 1/4),
        (1, 2, 4), (1, 3, 2), (2, 3, 1/3)
    ],
    'Площадь': [
        (0, 1, 1), (0, 2, 1/2), (0, 3, 1/3),
        (1, 2, 1/2), (1, 3, 1/3), (2, 3, 1/2)
    ],
    'Состояние': [
        (0, 1, 1/2), (0, 2, 1/2), (0, 3, 1/4),
        (1, 2, 1), (1, 3, 1/2), (2, 3, 1/2)
    ],
    'До учёбы': [
        (0, 1, 1/3), (0, 2, 1/5), (0, 3, 1/3),
        (1, 2, 1/2), (1, 3, 2), (2, 3, 3)
    ]
}

expert_1 = AHP(criteria, alternatives)
expert_1.add_criteria_comparison(criteria_comparisons_1)

for criterion in criteria:
    expert_1.add_alternative_comparison(criterion, apartment_comparisons[criterion])

print('Матрица сравнений критериев:')
display(pd.DataFrame(expert_1.criteria_matrix, index=criteria, columns=criteria).round(3))

for criterion in criteria:
    print('Сравнение квартир по критерию:', criterion)
    display(pd.DataFrame(expert_1.alternative_matrices[criterion],
                         index=alternatives, columns=alternatives).round(3))

Матрица сравнений критериев:


,Цена,До центра,До метро,Площадь,Состояние,До учёбы
Цена,1.000,9.0,4.0,9.0,9.0,4.0
До центра,0.111,1.0,0.5,1.0,1.0,0.5
До метро,0.250,2.0,1.0,2.0,2.0,1.0
Площадь,0.111,1.0,0.5,1.0,1.0,0.5
Состояние,0.111,1.0,0.5,1.0,1.0,0.5
До учёбы,0.250,2.0,1.0,2.0,2.0,1.0


Сравнение квартир по критерию: Цена


,Квартира А,Квартира Б,Квартира В,Квартира Г
Квартира А,1.000,4.0,2.000,3.0
Квартира Б,0.250,1.0,0.333,0.5
Квартира В,0.500,3.0,1.000,2.0
Квартира Г,0.333,2.0,0.500,1.0


Сравнение квартир по критерию: До центра


,Квартира А,Квартира Б,Квартира В,Квартира Г
Квартира А,1.0,0.200,0.333,0.5
Квартира Б,5.0,1.000,3.000,4.0
Квартира В,3.0,0.333,1.000,2.0
Квартира Г,2.0,0.250,0.500,1.0


... (4 outputs left out)

Сравнение квартир по критерию: Состояние


,Квартира А,Квартира Б,Квартира В,Квартира Г
Квартира А,1.0,0.5,0.5,0.25
Квартира Б,2.0,1.0,1.0,0.50
Квартира В,2.0,1.0,1.0,0.50
Квартира Г,4.0,2.0,2.0,1.00


Сравнение квартир по критерию: До учёбы


,Квартира А,Квартира Б,Квартира В,Квартира Г
Квартира А,1.0,0.333,0.200,0.333
Квартира Б,3.0,1.000,0.500,2.000
Квартира В,5.0,2.000,1.000,3.000
Квартира Г,3.0,0.500,0.333,1.000


## Задание 4. Расчёт весов, выбор квартиры и проверка согласованности

**Локальные веса** показывают предпочтительность квартиры по отдельному критерию. **Глобальные веса** учитывают сразу все критерии и их важность. Лучшим по оценкам эксперта считается вариант с наибольшим глобальным весом.

Для всех матриц проверяем CI и CR.

In [5]:
criteria_weights_1, local_weights_1, result_1 = expert_1.solve()

print('Веса критериев эксперта 1:')
display(pd.DataFrame({
    'Критерий': criteria,
    'Вес': criteria_weights_1
}).round(3))

print('Локальные веса квартир:')
display(pd.DataFrame(local_weights_1,
                     index=alternatives, columns=criteria).round(3))

print('Итоговый рейтинг квартир:')
rating_1 = pd.DataFrame({
    'Квартира': alternatives,
    'Глобальный вес': result_1
}).sort_values('Глобальный вес', ascending=False)
display(rating_1.round(3))

print('Лучшая квартира по оценкам эксперта 1:',
      alternatives[np.argmax(result_1)])

Веса критериев эксперта 1:


,Критерий,Вес
0,Цена,0.550
1,До центра,0.064
2,До метро,0.130
3,Площадь,0.064
4,Состояние,0.064
5,До учёбы,0.130


Локальные веса квартир:


,Цена,До центра,До метро,Площадь,Состояние,До учёбы
Квартира А,0.467,0.084,0.078,0.141,0.111,0.080
Квартира Б,0.095,0.546,0.492,0.141,0.222,0.270
Квартира В,0.277,0.232,0.125,0.263,0.222,0.477
Квартира Г,0.160,0.138,0.306,0.455,0.444,0.174


Итоговый рейтинг квартир:


,Квартира,Глобальный вес
0,Квартира А,0.299
2,Квартира В,0.276
3,Квартира Г,0.216
1,Квартира Б,0.209


Лучшая квартира по оценкам эксперта 1: Квартира А


In [6]:
print('Проверка согласованности экспертных оценок:')
all_matrices_1 = {'Критерии': expert_1.criteria_matrix}
all_matrices_1.update(expert_1.alternative_matrices)

consistency_rows = []
for name, matrix in all_matrices_1.items():
    ci, cr = expert_1.check_consistency(matrix)
    consistency_rows.append([name, ci, cr, 'Да' if cr < 0.1 else 'Нет'])

display(pd.DataFrame(consistency_rows,
                     columns=['Матрица', 'CI', 'CR', 'CR < 0.1?']).round(4))

if all(row[2] < 0.1 for row in consistency_rows):
    print('Все матрицы достаточно согласованы (CR < 0.1).')
else:
    print('Есть матрицы с недостаточной согласованностью.')

Проверка согласованности экспертных оценок:


,Матрица,CI,CR,CR < 0.1?
0,Критерии,0.0005,0.0004,Да
1,Цена,0.0103,0.0115,Да
2,До центра,0.0170,0.0189,Да
3,До метро,0.0161,0.0179,Да
4,Площадь,0.0035,0.0038,Да
5,Состояние,-0.0000,-0.0000,Да
6,До учёбы,0.0198,0.0220,Да


Все матрицы достаточно согласованы (CR < 0.1).


**Вывод по основным заданиям.** Для первого эксперта наибольший вес имеет цена (около 0.55). Поэтому он выбрал **квартиру А**: она самая дешёвая (60 000 руб. в месяц), хотя находится дальше от центра и до учёбы. У всех матриц **CR < 0.1**, значит, парные сравнения достаточно согласованы.

## Дополнительное задание. Учёт весов экспертов

Групповая оценка взвешенное среднее оценок экспертов:

$$a_{group} = \sum_{i=1}^{m} w_i a_i$$

Рассмотрим мнения двух экспертов:

- **Эксперт 1:** больше всего ценит низкую цену аренды. Вес его мнения — **0.6**.
- **Эксперт 2:** больше всего ценит короткую дорогу до учёбы, а также близость к центру и метро. Вес его мнения — **0.4**.

Они по-разному оценивают важность критериев, но одинаково сравнивают квартиры внутри каждого критерия. Посмотрим, как изменится общий выбор.

In [7]:
# Второй эксперт больше ценит дорогу до учёбы, центр и метро
criteria_comparisons_2 = [
    (0, 1, 1/2), (0, 2, 1/2), (0, 3, 2), (0, 4, 1), (0, 5, 1/3),
    (1, 2, 1), (1, 3, 4), (1, 4, 2), (1, 5, 1/2),
    (2, 3, 3), (2, 4, 2), (2, 5, 1/2),
    (3, 4, 1/2), (3, 5, 1/6),
    (4, 5, 1/3)
]

expert_2 = AHP(criteria, alternatives)
expert_2.add_criteria_comparison(criteria_comparisons_2)

# Квартиры по каждому критерию оба эксперта оценивают одинаково
for criterion in criteria:
    expert_2.add_alternative_comparison(criterion, apartment_comparisons[criterion])

criteria_weights_2, local_weights_2, result_2 = expert_2.solve()

print('Веса критериев двух экспертов:')
display(pd.DataFrame({
    'Критерий': criteria,
    'Эксперт 1': criteria_weights_1,
    'Эксперт 2': criteria_weights_2
}).round(3))

print('Лучшая квартира для эксперта 2:', alternatives[np.argmax(result_2)])

ci_2, cr_2 = expert_2.check_consistency(expert_2.criteria_matrix)
print(f'Согласованность критериев эксперта 2: CI = {ci_2:.4f}, CR = {cr_2:.4f}')
print('Оценки согласованы:', 'Да' if cr_2 < 0.1 else 'Нет')
print('Матрицы квартир у двух экспертов одинаковые, их CR проверены в задании 4.')

Веса критериев двух экспертов:


,Критерий,Эксперт 1,Эксперт 2
0,Цена,0.550,0.105
1,До центра,0.064,0.200
2,До метро,0.130,0.190
3,Площадь,0.064,0.055
4,Состояние,0.064,0.105
5,До учёбы,0.130,0.346


Лучшая квартира для эксперта 2: Квартира Б
Согласованность критериев эксперта 2: CI = 0.0037, CR = 0.0030
Оценки согласованы: Да
Матрицы квартир у двух экспертов одинаковые, их CR проверены в задании 4.


In [8]:
# Учитываем относительную важность мнений экспертов
expert_weights = np.array([0.6, 0.4])
expert_weights = expert_weights / expert_weights.sum()

group_result = expert_weights[0] * result_1 + expert_weights[1] * result_2

rating_group = pd.DataFrame({
    'Квартира': alternatives,
    'Эксперт 1': result_1,
    'Эксперт 2': result_2,
    'Групповая оценка': group_result
}).sort_values('Групповая оценка', ascending=False)

print('Сравнение индивидуальных и групповых оценок:')
display(rating_group.round(3))
print('Лучший вариант для группы:', alternatives[np.argmax(group_result)])

Сравнение индивидуальных и групповых оценок:


,Квартира,Эксперт 1,Эксперт 2,Групповая оценка
2,Квартира В,0.276,0.302,0.286
1,Квартира Б,0.209,0.337,0.260
0,Квартира А,0.299,0.127,0.230
3,Квартира Г,0.216,0.234,0.223


Лучший вариант для группы: Квартира В


Эксперт 1 выбрал **квартиру А**, потому что она самая дешёвая. Эксперт 2 предпочёл **квартиру Б** благодаря близости к центру и метро при сравнительно удобной дороге до учёбы. После объединения мнений с весами 0.6 и 0.4 лучшей стала **квартира В**: у неё умеренная стоимость и самое короткое время до учёбы (25 минут).

У экспертов разные предпочтения, поэтому их личные рейтинги отличаются. CR проверяет согласованность сравнений каждого эксперта, а не совпадение мнений экспертов.

### Вывод

Я использовала метод анализа иерархий Саати, чтобы сравнить 4 условные квартиры по 6 критериям. Составила матрицы парных сравнений, рассчитала локальные и глобальные веса, проверила CI и CR, определила лучший вариант для каждого эксперта и получила общий рейтинг с учётом весов экспертов.